# Comparing representations: what each rule keeps

Clustering decides **which** periods group together. Representation decides **what each
group's single profile looks like** — and it is a genuinely free choice: every clustering
method has a default, but any method can be paired with any of the six representations.

This tutorial puts all six on the **same cluster**, then compares the five concurrency
orderings within the distribution representation. We examine both the values kept and which
values occur together. By the end you should be able to look at a cluster of days and predict which
rule your model needs.

If you want the mechanics — how each rule computes its profile — see
[Representation](../explanation/how-aggregation-works/03_representation.ipynb). For a quick
recipe, see the [Representations how-to](../how-to/representations.ipynb).

## 1  A cluster to compare on

A representation rule acts on **one cluster at a time**, independently of every other cluster,
so a single cluster is the whole story.

We need one with **enough members to tell the rules apart** — with only two periods the medoid
and the maxoid tie and the mean sits exactly halfway between them. So we aggregate the
[tiny six-day set](../explanation/how-aggregation-works/01_preprocessing.ipynb) to **k=2**,
which gives Ward a four-member cluster to work with. That cluster also happens to contain the
series' 10 MW peak, which makes the differences easy to see.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, Distribution
from tsam.config import MinMaxMean
from tsam.plot import AttributeSpace

pio.renderers.default = "notebook_connected"

ATTRS = ["solar", "load"]
UNITS = {"solar": "W/m²", "load": "MW"}
N_TIMESTEPS = 4

tiny = pd.read_csv("../data/tiny.csv", index_col=0, parse_dates=True)

# Aggregate to k=2 so one cluster has four members. `preserve_column_means=False`
# turns off rescaling, so we see each rule's raw output rather than a corrected
# version of it (rescaling is a separate step, and a separate notebook).
baseline = tsam.aggregate(
    tiny,
    n_clusters=2,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    preserve_column_means=False,
)

# Pick the four-member cluster by *size*, never by label: cluster numbers are
# arbitrary names that can change between methods and runs.
focus = max(baseline.cluster_counts, key=baseline.cluster_counts.get)
members = [d for d, c in enumerate(baseline.cluster_assignments) if c == focus]

print("assignments, one per day:", list(baseline.cluster_assignments))
print(f"the cluster we follow has {len(members)} members: days {members}")

Those four days are what every rule below sees. Here they are in physical units —
the numbers each rule has to condense into one profile:

In [ ]:
def day_frame(series, days):
    """One row per day, columns (attribute, timestep), in physical units."""
    rows = {
        f"day{day}": {
            (attr, t): series[attr].iloc[day * N_TIMESTEPS + t]
            for attr in ATTRS
            for t in range(N_TIMESTEPS)
        }
        for day in days
    }
    frame = pd.DataFrame(rows).T
    frame.columns = pd.MultiIndex.from_tuples(frame.columns, names=["", "TimeStep"])
    return frame


members_physical = day_frame(tiny, members)
print("The cluster's four members — solar in W/m², load in MW:")
members_physical.round(2)

## 2  The cluster, drawn

A period is not a point — it is a **path**. With two attributes we can draw it directly: solar
on one axis, load on the other, one marker per timestep. Each day traces a path from `t0` to
`t3`, and the arrows keep the direction of travel visible.

Read the four days as a family: all start and end at `solar = 0` (night), swing right as the
sun comes up, and sit at different heights on the load axis. **day5** is the outlier — it
barely leaves the load axis and climbs to 10 MW.

In [ ]:
space = AttributeSpace(
    "solar",
    "load",
    units=UNITS,
    title="The four members of the cluster — each a path from t0 to t3",
)
for day in members:
    block = tiny.iloc[day * N_TIMESTEPS : (day + 1) * N_TIMESTEPS]
    space.add_path(block["solar"], block["load"], name=f"day{day}")
space.show()

## 3  The six rules, one call each

The `representation=` lever takes the six names below. Five are plain strings; `MinMaxMean` is
a typed object, because a bare string cannot carry the extra choice it needs — *which* column
gets the max and which gets the mean. Here `load` is set to `max`: an envelope on demand, an
average on supply.

Everything else is held fixed, so any difference in the output is the representation's doing:

In [ ]:
rules = {
    "mean": "mean",
    "medoid": "medoid",
    "maxoid": "maxoid",
    "distribution": "distribution",
    "distribution_minmax": "distribution_minmax",
    "minmax_mean": MinMaxMean(max_columns=["load"], min_columns=[]),
}

profiles = {}
for name, representation in rules.items():
    result = tsam.aggregate(
        tiny,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )
    # The grouping is identical every time — only the profile changes — but the
    # label can move, so find our cluster by its members rather than its number.
    cid = int(result.cluster_assignments[members[0]])
    profiles[name] = result.cluster_representatives.loc[cid]

print("Six representatives of the same four days (physical units):")
pd.concat(profiles, names=["rule"]).round(2)

## 4  Three questions to ask of each rule

**Is the profile a day that actually happened?** Only a rule that lifts a whole member out of
the cluster can promise that — and with it, every internal correlation the day had (solar and
load at the same hour). A rule that builds the profile timestep by timestep cannot.

**Did the cluster's 10 MW peak survive?** That is what a capacity-sizing model needs.

**Does the profile still carry the cluster's average?** That is what an energy-balance model
needs — and it is the property that decides whether the aggregation has to be
[rescaled](../explanation/how-aggregation-works/05_rescaling.ipynb) afterwards.

In [ ]:
# What the cluster really contains, against which each rule is scored.
cluster_block = pd.concat(
    [tiny.iloc[d * N_TIMESTEPS : (d + 1) * N_TIMESTEPS] for d in members]
)
true_peak = cluster_block["load"].max()
true_means = {a: cluster_block[a].mean() for a in ATTRS}

rows = []
for name, profile in profiles.items():
    values = np.concatenate([profile[a].to_numpy() for a in ATTRS])
    match = next(
        (
            day
            for day in members
            if np.allclose(
                values,
                np.concatenate(
                    [members_physical.loc[f"day{day}", a].to_numpy() for a in ATTRS]
                ),
                atol=1e-9,
            )
        ),
        None,
    )
    keeps_means = all(
        np.isclose(profile[a].mean(), true_means[a], atol=1e-12) for a in ATTRS
    )
    rows.append(
        {
            "rule": name,
            "a real day?": f"yes — day{match}" if match is not None else "constructed",
            "load peak [MW]": profile["load"].max(),
            "keeps the peak?": "yes"
            if np.isclose(profile["load"].max(), true_peak)
            else "no",
            "mean load [MW]": profile["load"].mean(),
            "keeps both means?": "yes" if keeps_means else "no",
        }
    )

means_text = ", ".join(f"{a} {m:.4g}" for a, m in true_means.items())
print(
    f"The cluster's true load peak is {true_peak:.0f} MW; its true means are {means_text}.\n"
)
pd.DataFrame(rows).set_index("rule").round(2)

Read down the table and the trade-off is complete — and notice that **no rule scores
yes on everything**:

* **`medoid` and `maxoid` return a period that really happened.** One whole member is lifted
  out of the cluster, so the day stays internally consistent. The price is that a single day
  stands in for four, and its average is whatever that day's average happened to be. Which day
  you get depends on where the rule looks: the medoid looks **inward** (closest to its
  cluster-mates, so the most typical) and misses the peak; the maxoid looks **outward**
  (furthest from the rest of the series, so the most extreme) and keeps it.
* **`mean` flattens the peak to 7 MW.** Each of the eight numbers is averaged independently, so
  no member's day survives — and averaging is precisely what destroys extremes. This is what
  [extreme periods](../explanation/how-aggregation-works/04_extreme_periods.ipynb) exists to fix.
* **`distribution` keeps the means *and* gets closer on the peak.** It sorts the cluster's
  pooled values and averages them into fewer levels. This preserves the average and
  approximates the duration curve; it does not reproduce every original value.
* **`distribution_minmax` recovers the full 10 MW and keeps the means here.** It pins the
  extremes and redistributes the change across interior levels. This preserves the integral
  when the bounds and number of timesteps allow it.
* **`minmax_mean` keeps the peak and the solar mean, but not the load mean** — because we asked
  for `load: max`. Every load timestep is now an envelope rather than a plausible day.

Here `mean`, `distribution`, and `distribution_minmax` pass **keeps both means**. The other
rules change at least one total in this example, which is why tsam offers rescaling. For
`distribution_minmax`, retaining both the mean and extremes depends on feasibility.

## 5  Real days land on a member's path

The split between "a real day" and "constructed" is visible directly. `medoid` and `maxoid`
land **exactly on a member's path**, because they *are* that member. The other four trace paths
**no day ever took** — they are assembled timestep by timestep.

Only the first guarantees a day that was actually observed before rescaling. Constructed
profiles may still be useful, but their cross-attribute relationships need checking. Click a
legend entry to isolate one rule.

In [ ]:
space = AttributeSpace(
    "solar",
    "load",
    units=UNITS,
    title="Members (grey) and the six representatives — click the legend to isolate one",
)
for day in members:
    block = tiny.iloc[day * N_TIMESTEPS : (day + 1) * N_TIMESTEPS]
    space.add_path(
        block["solar"],
        block["load"],
        name=f"day{day}",
        color="#c9c9c9",
        symbol="circle",
        label_steps=False,
        arrows=False,
    )

# A distinct symbol per rule, so the six stay apart even where they overlap.
symbols = ["circle", "square", "diamond", "triangle-up", "x", "star"]
for (name, profile), symbol in zip(profiles.items(), symbols):
    space.add_path(
        profile["solar"],
        profile["load"],
        name=name,
        symbol=symbol,
        width=3,
        dash="dot",
        label_steps=False,
    )
space.show()

## 6  Same duration curves, different co-occurrence

So far `distribution` has used its default ordering: each attribute's levels follow the
ranks of that attribute's mean profile. The `Distribution` object lets us change that last
step. These are **five orderings of one representation**, not five new clustering methods.

We keep the same six days, two clusters, and four-member focus cluster. Before running the
next cell, predict which property should stay fixed when we only permute a profile's values:
its peak, its mean, or the solar/load pairs at each timestep?

`medoid` borrows separate ranks for each attribute from one real member. `reference` borrows
one column's mean-profile ranks for every attribute; it needs the name of that column.
`consensus` derives shared ranks from the first principal component of the standardized mean
profiles. `assignment` finds the shared ordering with the smallest total squared deviation
from the cluster mean profile.

In [ ]:
orderings = {
    "independent": Distribution(),
    "medoid": Distribution(concurrency="medoid"),
    "reference": Distribution(concurrency="reference", reference_attribute="solar"),
    "consensus": Distribution(concurrency="consensus"),
    "assignment": Distribution(concurrency="assignment"),
}

ordered_results = {}
ordered_profiles = {}
for name, representation in orderings.items():
    result = tsam.aggregate(
        tiny,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )
    cid = int(result.cluster_assignments[members[0]])
    assert set(np.flatnonzero(result.cluster_assignments == cid)) == set(members)
    ordered_results[name] = result
    ordered_profiles[name] = result.cluster_representatives.loc[cid]

pd.concat(ordered_profiles, names=["ordering"]).round(2)

### Check what stayed fixed

Sort each attribute's four representative values. Every strategy must give the same list,
so the mean, minimum, and maximum must also agree. This check concerns the **compressed**
duration curves; they still approximate the original data.

In [ ]:
sorted_profiles = {
    name: pd.DataFrame(np.sort(profile[ATTRS].to_numpy(), axis=0), columns=ATTRS)
    for name, profile in ordered_profiles.items()
}
for profile in sorted_profiles.values():
    np.testing.assert_allclose(profile, sorted_profiles["independent"])

pd.concat(sorted_profiles, names=["ordering", "ascending rank"]).round(2)

### Draw what changed

The solar/load pairs can change even though both sorted lists are identical. Compare the
paths below with the real medoid's path (grey). The `medoid` ordering borrows that day's ranks,
but replaces its values with duration-curve levels, so it does **not** return the real day.
Tied values also mean rank ordering cannot guarantee exact rank correlations.

The `reference`, `consensus`, and `assignment` strategies all pair low solar with low load
and high solar with high load within the period. Their paths visit the same pairs, even when
they visit them at different timesteps. This can impose positive correlation that the member
days did not have. Use the legend to inspect one strategy at a time.

In [ ]:
space = AttributeSpace(
    "solar",
    "load",
    units=UNITS,
    title="Same duration-curve values, different solar/load pairs",
)
space.add_path(
    profiles["medoid"]["solar"],
    profiles["medoid"]["load"],
    name="real medoid",
    color="#999999",
)
for (name, profile), symbol in zip(ordered_profiles.items(), symbols):
    space.add_path(
        profile["solar"],
        profile["load"],
        name=name,
        symbol=symbol,
        dash="dot",
        label_steps=True,
    )
space.show()

### Measure the whole reconstruction

The plot follows one cluster; the metrics below cover all six reconstructed days.
`result.concurrency` reports the difference between original and reconstructed correlation
matrices: Pearson for linear relationships, Spearman for rank relationships. Both use the
Frobenius norm (the square root of the sum of squared matrix differences); lower is better.
They summarize pairwise relationships, not every aspect of joint behavior or chronology.

In [ ]:
concurrency_scores = pd.DataFrame(
    {
        name: {
            "RMSE (timing)": result.accuracy.weighted_rmse,
            "RMSE (duration curve)": result.accuracy.weighted_rmse_duration,
            "Pearson correlation error": result.concurrency.correlation_error,
            "Spearman correlation error": result.concurrency.rank_correlation_error,
        }
        for name, result in ordered_results.items()
    }
).T

np.testing.assert_allclose(
    concurrency_scores["RMSE (duration curve)"],
    concurrency_scores.loc["independent", "RMSE (duration curve)"],
)
concurrency_scores.round(4)

The duration-curve errors agree. On these six days, `independent` has smaller timing and
correlation errors than `medoid`. Borrowing a real medoid's ranks does not guarantee the
best score; the hourly-data example in the how-to shows a case where `medoid` helps.
Shared-order strategies can have identical correlation errors because moving their paired
values together does not change correlations, even when it changes timing error.

Repeat this comparison on your own attributes and check the relationships your model uses.
These examples use local cluster distributions without segmentation or rescaling to isolate
the ordering choice. For configuration limits and combining concurrency with min/max
preservation, see the
[how-to recipe](../how-to/representations.ipynb#choose-how-attributes-coincide).

## 7  Which to use

There is no default that is right for every model. Pick the property you cannot afford to lose:

| If your model… | Use | Because |
|---|---|---|
| needs physically consistent days (correlated attributes) | `medoid` | it is a real day, and the most typical one |
| sizes capacity against the worst case | `maxoid` or extreme periods | it is a real day, and the most extreme one |
| only cares about totals and averages | `mean` | the simplest rule that preserves the cluster's mean |
| is sensitive to the duration curve | `distribution` | it approximates how often values occur and keeps the mean |
| needs duration curves and relationships between attributes | `Distribution(concurrency="medoid")` as a candidate | it keeps the same marginal fit while borrowing a real day's ranks; check the concurrency metrics |
| needs the duration curve **and** a hard envelope | `distribution_minmax` | it pins the true min and max into the profile |
| needs a guaranteed bound on specific columns | `minmax_mean` | you choose per column: min, max or mean |

The distribution orderings are available for `scope="local"` on `ClusterConfig`, before any
segmentation. A good duration-curve fit alone does not validate storage behavior: also check
chronology and the relationships between supply and demand.

`mean` and `distribution` preserve cluster means; `distribution_minmax` also does so when
feasible. Other rules can change totals. tsam **rescales** representatives by default
(`preserve_column_means=True`), which we turned off here to see the raw rules. Rescaling can
also change the profiles, so assess the final configuration you will use; see
[Rescaling](../explanation/how-aggregation-works/05_rescaling.ipynb).

---

**Where to go next**

* [Representation](../explanation/how-aggregation-works/03_representation.ipynb) — how each rule
  computes its profile, worked by hand.
* [Comparing clustering methods](comparing_clustering_methods.ipynb) — the other axis: same
  representation, different groupings.
* [Choosing a method](choosing_a_method.ipynb) — all four levers together, and how to read a
  configuration off your model.
* [Extreme periods](../explanation/how-aggregation-works/04_extreme_periods.ipynb) — the other
  way to keep a peak, and why it is not interchangeable with `maxoid`.